## Understand the embedding

In [42]:
from keras.layers import CategoryEncoding, TextVectorization

In [43]:
## demo sentences
sent = [ 'the plate of rice',
         'the bowl of soup',
         'the bottle of water',
         'she is a kind girl',
         'he is a smart teacher',
         'learn the basics of coding',
         'their projects are impressive']


In [44]:
sent

['the plate of rice',
 'the bowl of soup',
 'the bottle of water',
 'she is a kind girl',
 'he is a smart teacher',
 'learn the basics of coding',
 'their projects are impressive']

In [45]:
## Define the maximum vocabulary size
voc_size = 10000

In [46]:
## Convert text to integer token ids 
vectorizer = TextVectorization(max_tokens=voc_size, output_mode="int")
vectorizer.adapt(sent)
tokenized_sent = vectorizer(sent)
tokenized_sent

<tf.Tensor: shape=(7, 5), dtype=int64, numpy=
array([[ 2, 14,  3, 12,  0],
       [ 2, 21,  3,  9,  0],
       [ 2, 22,  3,  6,  0],
       [11,  4,  5, 16, 19],
       [18,  4,  5, 10,  8],
       [15,  2, 23,  3, 20],
       [ 7, 13, 24, 17,  0]])>

In [47]:
## Convert to multihot encoded representation
mhe = CategoryEncoding(num_tokens=voc_size, output_mode="multi_hot")
mhe_representation = mhe(tokenized_sent)
mhe_representation

<tf.Tensor: shape=(7, 10000), dtype=float32, numpy=
array([[1., 0., 1., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], shape=(7, 10000), dtype=float32)>

### Word embedding layer

In [48]:
from keras.layers import Embedding
from keras.utils import pad_sequences
from keras.models import Sequential

import numpy as np

In [49]:
## Make all sentences as equal size
max_sent_length = 8
padded_docs = pad_sequences(
    sequences=tokenized_sent,
    padding='pre',
    maxlen=max_sent_length
)
padded_docs

array([[ 0,  0,  0,  2, 14,  3, 12,  0],
       [ 0,  0,  0,  2, 21,  3,  9,  0],
       [ 0,  0,  0,  2, 22,  3,  6,  0],
       [ 0,  0,  0, 11,  4,  5, 16, 19],
       [ 0,  0,  0, 18,  4,  5, 10,  8],
       [ 0,  0,  0, 15,  2, 23,  3, 20],
       [ 0,  0,  0,  7, 13, 24, 17,  0]], dtype=int32)

#### Comment:
The above result shows each sentences with equal size.

In [50]:
## Feature representation
dim=10  # dimension of feature vector

In [ ]:
model = Sequential()
model.add(Embedding(
    input_dim=voc_size,     # size of vocabulary(maximum integer index + 1)
    output_dim=dim,
    ))
model.build(input_shape=(None, max_sent_length))    # (batch_size, length_of_sent_vector)
model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 8, 10)          │       100,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,000 (390.62 KB)

 Trainable params: 100,000 (390.62 KB)

 Non-trainable params: 0 (0.00 B)

In [52]:
## Compile the model
model.compile('adam', 'mse')

In [54]:
## See the embedded docs
model.predict(padded_docs)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step


array([[[-0.04732496,  0.04103759,  0.03798858,  0.03612913,
         -0.02279869, -0.00987422,  0.04389432, -0.02896936,
          0.02677884,  0.01941354],
        [-0.04732496,  0.04103759,  0.03798858,  0.03612913,
         -0.02279869, -0.00987422,  0.04389432, -0.02896936,
          0.02677884,  0.01941354],
        [-0.04732496,  0.04103759,  0.03798858,  0.03612913,
         -0.02279869, -0.00987422,  0.04389432, -0.02896936,
          0.02677884,  0.01941354],
        [-0.02718066,  0.00100336,  0.02506315,  0.02856933,
         -0.01050232,  0.03826097, -0.03957133, -0.01151311,
          0.03397224,  0.00323821],
        [-0.03279524, -0.04599131,  0.01619157, -0.02916365,
          0.0147994 , -0.01610605, -0.00456257, -0.00064441,
          0.01808895,  0.01041529],
        [-0.02522752,  0.04229522, -0.01570238,  0.03597379,
          0.04308524,  0.04655102, -0.00093635, -0.00073066,
         -0.03507341,  0.04639452],
        [ 0.0093807 ,  0.01619518, -0.03244791, -0.0

In [55]:
## Original first sentence vector
padded_docs[0]

array([ 0,  0,  0,  2, 14,  3, 12,  0], dtype=int32)

In [56]:
## After embedding
model.predict(padded_docs[0].reshape(1,-1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


array([[[-0.04732496,  0.04103759,  0.03798858,  0.03612913,
         -0.02279869, -0.00987422,  0.04389432, -0.02896936,
          0.02677884,  0.01941354],
        [-0.04732496,  0.04103759,  0.03798858,  0.03612913,
         -0.02279869, -0.00987422,  0.04389432, -0.02896936,
          0.02677884,  0.01941354],
        [-0.04732496,  0.04103759,  0.03798858,  0.03612913,
         -0.02279869, -0.00987422,  0.04389432, -0.02896936,
          0.02677884,  0.01941354],
        [-0.02718066,  0.00100336,  0.02506315,  0.02856933,
         -0.01050232,  0.03826097, -0.03957133, -0.01151311,
          0.03397224,  0.00323821],
        [-0.03279524, -0.04599131,  0.01619157, -0.02916365,
          0.0147994 , -0.01610605, -0.00456257, -0.00064441,
          0.01808895,  0.01041529],
        [-0.02522752,  0.04229522, -0.01570238,  0.03597379,
          0.04308524,  0.04655102, -0.00093635, -0.00073066,
         -0.03507341,  0.04639452],
        [ 0.0093807 ,  0.01619518, -0.03244791, -0.0